In [16]:
# Lab Assignment 6 - DSA Cleaned Submissions dataset
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"
print("Plotly version:", __import__("plotly").__version__)


# 1. Load and inspect the data
df = pd.read_csv("DSA_Cleaned_Submissions-1.csv")
print("Dataset shape:", df.shape)
print("\\nColumns:", df.columns.tolist())
display(df.head())

# Data types and missing values
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2)
})
display(summary)

# Summary statistics for the numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
display(df[numeric_cols].describe().round(2))


# 2. Cleaning and feature preparation

required = ["programming_activity", "timestamp_utc", "outcome", "attempt_version", "success_binary"]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True, errors="coerce")
df["programming_activity"] = df["programming_activity"].fillna("Unknown activity").astype(str)
df["outcome"] = df["outcome"].fillna("Empty / unclassified").astype(str).str.strip()
df["attempt_version"] = pd.to_numeric(df["attempt_version"], errors="coerce")
df["success_binary"] = pd.to_numeric(df["success_binary"], errors="coerce")

df = df.dropna(subset=["timestamp_utc", "attempt_version"]).copy()
df["year"] = df["timestamp_utc"].dt.year
df["month"] = df["timestamp_utc"].dt.to_period("M").astype(str)
df["date"] = df["timestamp_utc"].dt.date
df = df.sort_values("timestamp_utc").reset_index(drop=True)

OUTCOME_ORDER = ["Success", "Failure", "Empty / unclassified"]
observed_outcomes = df["outcome"].value_counts().reindex(OUTCOME_ORDER).dropna()
print("Rows after basic cleaning:", len(df))
print("\\nOutcome counts:")
display(observed_outcomes.to_frame("submissions"))
print("\\nDate range:", df["timestamp_utc"].min(), "to", df["timestamp_utc"].max())


# Task 1: Core Interactive Charts

#Univariate Histogram
fig1 = px.histogram(
    df, x="attempt_version",
    title="Distribution of Attempt Versions in DSA Submissions",
    labels={"attempt_version": "Attempt version"}
)
fig1.update_layout(
    xaxis_title="Attempt version",
    yaxis_title="Number of submissions",
    bargap=0.08,
    xaxis=dict(rangeslider=dict(visible=True, thickness=0.12))
)
fig1.show()

# Multivariate Scatter

fig2 = px.scatter(
    df, x="timestamp_utc", y="attempt_version",
    color="outcome",
    hover_data=["programming_activity", "user_id", "success_binary"],
    opacity=0.65,
    labels={
        "timestamp_utc": "Submission timestamp (UTC)",
        "attempt_version": "Attempt version",
        "outcome": "Outcome"
    },
    title="DSA Submission Timeline: Attempt Version and Outcome"
)
fig2.show()


# Box plot

fig3 = px.box(
    df, x="outcome", y="attempt_version", color="outcome",
    points="outliers",
    category_orders={"outcome": OUTCOME_ORDER},
    hover_data=["programming_activity", "success_binary"],
    labels={"outcome": "Outcome", "attempt_version": "Attempt version"},
    title="Attempt Version Distribution by Submission Outcome"
)
fig3.update_layout(showlegend=False)
fig3.show()

# Median and quartiles behind the box plot
display(df.groupby("outcome")["attempt_version"].agg(
    submissions="count", median="median", mean="mean",
    q1=lambda x: x.quantile(0.25), q3=lambda x: x.quantile(0.75)
).round(2))

# 3D scatter
activity_stats = df.groupby("programming_activity").agg(
    submissions=("submission_id", "count"),
    success_rate=("success_binary", "mean"),
    mean_attempt_version=("attempt_version", "mean")
).reset_index()
activity_stats["success_rate_pct"] = activity_stats["success_rate"] * 100
activity_stats["failure_rate_pct"] = (1 - activity_stats["success_rate"]) * 100
activity_stats["success_rate_pct"] = activity_stats["success_rate_pct"].round(2)
activity_stats["failure_rate_pct"] = activity_stats["failure_rate_pct"].round(2)

fig4 = px.scatter_3d(
    activity_stats, x="submissions", y="success_rate_pct", z="mean_attempt_version",
    color="failure_rate_pct", size="submissions",
    hover_name="programming_activity",
    labels={
        "submissions": "Submission count",
        "success_rate_pct": "Success rate (%)",
        "mean_attempt_version": "Mean attempt version",
        "failure_rate_pct": "Failure rate (%)"
    },
    title="Programming Activity Performance in 3D"
)
fig4.update_layout(height=650)
fig4.show()

# Task 2: Animation and Dropdown Controls

top_activities = (
    df["programming_activity"].value_counts().head(15).index.tolist()
)
year_activity = (
    df[df["programming_activity"].isin(top_activities)]
    .groupby(["year", "programming_activity"])
    .agg(
        submissions=("submission_id", "count"),
        success_rate=("success_binary", "mean"),
        mean_attempt_version=("attempt_version", "mean")
    )
    .reset_index()
)
year_activity["success_rate_pct"] = year_activity["success_rate"] * 100

# Complete the year/activity grid so activities remain represented consistently.
all_years = sorted(year_activity["year"].unique())
full_index = pd.MultiIndex.from_product(
    [all_years, top_activities], names=["year", "programming_activity"]
)
year_activity = (
    year_activity.set_index(["year", "programming_activity"])
    .reindex(full_index)
    .reset_index()
)
year_activity["submissions"] = year_activity["submissions"].fillna(0)
# Rates remain missing when an activity has no submissions in that year.
year_activity["success_rate_pct"] = year_activity["success_rate_pct"].fillna(0)
year_activity["mean_attempt_version"] = year_activity["mean_attempt_version"].fillna(0)

# Animated Scatter
fig5 = px.scatter(
    year_activity, x="submissions", y="success_rate_pct",
    size="mean_attempt_version", color="programming_activity",
    animation_frame="year", animation_group="programming_activity",
    hover_name="programming_activity",
    range_x=[0, max(1, year_activity["submissions"].max() * 1.1)],
    range_y=[0, 100], size_max=35,
    labels={
        "submissions": "Number of submissions",
        "success_rate_pct": "Success rate (%)",
        "programming_activity": "Programming activity",
        "mean_attempt_version": "Mean attempt version"
    },
    title="DSA Activity Performance Over Time (by year)"
)
fig5.update_layout(legend_title_text="Programming activity")
if fig5.layout.updatemenus:
    fig5.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 800
fig5.show()

# Dropdown to switch the activity metric


metric_df = activity_stats.nlargest(15, "submissions").sort_values("submissions")
metric_df = metric_df.copy()
metric_df["success_rate_pct"] = metric_df["success_rate"] * 100

metric_options = {
    "Submission count": ("submissions", "Submission count"),
    "Success rate (%)": ("success_rate_pct", "Success rate (%)"),
    "Mean attempt version": ("mean_attempt_version", "Mean attempt version")
}
initial_col = "submissions"
fig6 = go.Figure()
fig6.add_trace(go.Bar(
    x=metric_df[initial_col],
    y=metric_df["programming_activity"],
    orientation="h",
    marker_color="#3366cc",
    hovertemplate="<b>%{y}</b><br>Value: %{x:.2f}<extra></extra>"
))

buttons = []
for button_label, (col, axis_label) in metric_options.items():
    buttons.append(dict(
        label=button_label,
        method="update",
        args=[
            {"x": [metric_df[col]]},
            {"xaxis": {"title": axis_label},
             "title": f"Top 15 DSA Activities: {axis_label}"}
        ]
    ))

fig6.update_layout(
    title="Top 15 DSA Activities: Submission count",
    xaxis_title="Submission count",
    yaxis_title="Programming activity",
    yaxis=dict(rangemode="tozero"),
    updatemenus=[dict(
        buttons=buttons, direction="down", x=1.0, xanchor="right",
        y=1.18, yanchor="top", showactive=True
    )],
    margin=dict(l=220, t=100),
    height=650
)
fig6.show()

# Spearman correlation is calculated only for rows with a known binary success value.
corr_data = df[["attempt_version", "success_binary"]].dropna()
if len(corr_data) > 1 and corr_data["success_binary"].nunique() > 1:
    print("Spearman correlation (attempt version vs success flag):",
          round(corr_data.corr(method="spearman").iloc[0, 1], 3))
else:
    print("Not enough variation in success_binary to calculate a meaningful correlation.")


# --- Comparison: static chart vs. animation ---

figs = [fig1, fig2, fig3, fig4, fig5, fig6]
with open("Lab6_DSA_interactive_charts.html", "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><title>Lab 6: DSA interactive charts</title></head><body>")
    for i, fig in enumerate(figs):
        f.write(fig.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False,
                            auto_play=False))
    f.write("</body></html>")
print("Saved Lab6_DSA_interactive_charts.html")

Plotly version: 5.24.1
Dataset shape: (15693, 8)
\nColumns: ['submission_id', 'user_id', 'attempt_version', 'programming_activity', 'timestamp_utc', 'result_raw', 'outcome', 'success_binary']


,submission_id,user_id,attempt_version,programming_activity,timestamp_utc,result_raw,outcome,success_binary
0,0,user_0001,1,negative_counting_sort,2017-06-01 13:27:37 UTC,success: ....,Success,1
1,1,user_0001,1,order_statistic,2017-06-01 13:38:28 UTC,success: ....,Success,1
2,2,user_0002,1,order_statistic,2017-06-02 01:53:02 UTC,success: ....,Success,1
3,3,user_0003,1,order_statistic,2017-06-02 01:58:25 UTC,success: ....,Success,1
4,4,user_0003,1,negative_counting_sort,2017-06-02 01:59:31 UTC,success: ....,Success,1


,dtype,missing,missing_%
submission_id,int64,0,0.0
user_id,object,0,0.0
attempt_version,int64,0,0.0
programming_activity,object,0,0.0
timestamp_utc,object,0,0.0
result_raw,object,0,0.0
outcome,object,0,0.0
success_binary,int64,0,0.0


,submission_id,attempt_version,success_binary
count,15693.00,15693.00,15693.00
mean,8044.70,5.82,0.36
std,4646.72,9.25,0.56
min,0.00,1.00,-1.00
25%,4013.00,1.00,0.00
50%,8076.00,3.00,0.00
75%,12079.00,6.00,1.00
max,16073.00,104.00,1.00


Rows after basic cleaning: 15693
\nOutcome counts:


/tmp/ipykernel_2363/2734794340.py:46: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.



,submissions
outcome,
Success,6321
Failure,8772
Empty / unclassified,600


\nDate range: 2017-05-16 14:20:33+00:00 to 2024-09-12 02:51:12+00:00


,submissions,median,mean,q1,q3
outcome,,,,,
Empty / unclassified,600,10.0,18.21,4.0,20.0
Failure,8772,3.0,6.47,2.0,8.0
Success,6321,2.0,3.74,1.0,4.0


Spearman correlation (attempt version vs success flag): -0.292
Saved Lab6_DSA_interactive_charts.html
